# Momentum Seasonality Analyse
Dit notebook onderzoekt de seizoensinvloeden van verschillende Momentum ETF's in vergelijking met de brede markt (S&P 500).

In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Stel de stijl in voor de grafieken
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

### 1. Data Downloaden
We halen de historische data op van MTUM, QMOM, DWAS, IWMO.AS en SPY.

In [ ]:
# Definieer de ETF's en de benchmark
tickers = {
    'MTUM': 'US Large Cap Momentum',
    'QMOM': 'US Quantitative Momentum',
    'DWAS': 'US Small Cap Momentum',
    'IWMO.AS': 'Global Momentum',
    'SPY': 'S&P 500 (Benchmark)'
}

# Download data via yfinance (Vanaf 2015 omdat sommige ETF's toen pas lanceerden)
print("Data downloaden...")
data = yf.download(list(tickers.keys()), start="2015-01-01")

# Pak alleen de 'Adj Close' (gecorrigeerde slotkoers)
if 'Adj Close' in data.columns:
    data = data['Adj Close']
elif isinstance(data.columns, pd.MultiIndex):
    data = data.xs('Adj Close', level=0, axis=1)

# Ontbrekende waarden in het begin weglaten
data.dropna(how='all', inplace=True)
display(data.tail())

### 2. Maandelijkse Rendementen Berekenen
We converteren de dagelijkse koersen naar maandelijkse rendementen.

In [ ]:
# Resample naar de laatste dag van de maand
monthly_data = data.resample('ME').last()

# Bereken het procentuele rendement
monthly_returns = monthly_data.pct_change().dropna(how='all')

# Voeg Jaar en Maand kolommen toe om makkelijk te groeperen in een pivot table
monthly_returns['Year'] = monthly_returns.index.year
monthly_returns['Month'] = monthly_returns.index.month

display(monthly_returns.head())

In [ ]:
def plot_seasonality_heatmap(ticker_symbol, title):
    if ticker_symbol not in monthly_returns.columns:
        print(f"Geen data gevonden voor {ticker_symbol}")
        return
        
    # Creëer een pivot table voor de heatmap (Jaar op Y-as, Maand op X-as)
    pivot_table = monthly_returns.pivot_table(values=ticker_symbol, index='Year', columns='Month')
    
    # Definieer maandnamen (en let op dat we alleen de namen pakken die in de kolommen zitten)
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    pivot_table.columns = month_names[:len(pivot_table.columns)]
    
    plt.figure(figsize=(12, 6))
    # Plot met seaborn
    sns.heatmap(pivot_table * 100, annot=True, fmt=".1f", cmap="RdYlGn", center=0, 
                cbar_kws={'label': 'Rendement (%)'}, linewidths=.5)
    plt.title(f'Seasonality Heatmap: {title} ({ticker_symbol})', fontsize=16)
    plt.yticks(rotation=0)
    plt.show()

### 3. Heatmaps Genereren
Laten we kijken naar de seizoenspatronen voor de verschillende ETF's.

In [ ]:
# S&P 500 (Benchmark)
plot_seasonality_heatmap('SPY', tickers['SPY'])

In [ ]:
# US Large Cap Momentum
plot_seasonality_heatmap('MTUM', tickers['MTUM'])

In [ ]:
# US Quant/Aggressive Momentum
plot_seasonality_heatmap('QMOM', tickers['QMOM'])

In [ ]:
# US Small Cap Momentum (Januari effect?)
plot_seasonality_heatmap('DWAS', tickers['DWAS'])

### 4. Gemiddelde Rendementen Vergelijken

In [ ]:
# Bereken de gemiddelde rendementen over alle jaren per maand
mean_returns = monthly_returns.drop('Year', axis=1).groupby('Month').mean() * 100
mean_returns.index = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Plot een bar chart voor vergelijking
mean_returns[['MTUM', 'QMOM', 'DWAS', 'SPY']].plot(kind='bar', figsize=(14, 7), width=0.8)
plt.title('Gemiddeld Maandrendement per ETF', fontsize=16)
plt.ylabel('Gemiddeld Rendement (%)')
plt.xlabel('Maand')
plt.axhline(0, color='black', linewidth=1)
plt.legend(title='ETF')
plt.grid(axis='y', alpha=0.3)
plt.show()